# Modelado - Enfoque A: Validación Estática (Hold-out)
En esta notebook utilizaremos la estrategia clásica de validación estática, donde hemos separado un 70% para entrenamiento y un 15% para validación (dejando el 15% final para test).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

### 1. Carga de Datos

In [ ]:
base_path = '../data/model_input/static/'

X_train = pd.read_csv(base_path + 'X_train.csv')
y_train = pd.read_csv(base_path + 'y_train.csv').values.ravel()

X_val = pd.read_csv(base_path + 'X_val.csv')
y_val = pd.read_csv(base_path + 'y_val.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

### 2. Definición del Preprocesador (ColumnTransformer)

In [ ]:
vars_categoricas = [
    'meal', 'market_segment', 'distribution_channel', 
    'reserved_room_type', 'deposit_type', 'customer_type'
]

vars_numericas = [
    'lead_time', 'adr', 'total_nights', 'adults', 'children', 'babies', 
    'previous_cancellations', 'required_car_parking_spaces', 
    'total_of_special_requests', 'agent', 'company', 
    'arrival_date_year', 'arrival_month_num' 
]

vars_binarias = [
    'is_repeated_guest', 'is_placed_on_waiting_list', 'is_portugal', 'is_resort'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), vars_numericas),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), vars_categoricas),
        ('bin', 'passthrough', vars_binarias)
    ], 
    remainder='drop'
)

### 3. Creación del Pipeline y Entrenamiento

In [ ]:
# Instanciamos el modelo con class_weight='balanced' para manejar el desbalanceo
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

pipeline_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', lr_model)
])

print("Entrenando el modelo rf...")
pipeline_rf.fit(X_train, y_train)
print("¡Entrenamiento completado!")

print("Entrenando el modelo lr...")
pipeline_lr.fit(X_train, y_train)
print("¡Entrenamiento completado!")

### 4. Evaluación en el Set de Validación

In [ ]:
y_pred = pipeline.predict(X_val)
y_pred_proba = pipeline.predict_proba(X_val)[:, 1]

print("=== Reporte de Clasificación (Validación Estática) ===")
print(classification_report(y_val, y_pred))

roc_auc = roc_auc_score(y_val, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Matriz de confusión
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Cancelado', 'Cancelado'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Matriz de Confusión - Validación Estática')
plt.show()

In [ ]:
def evaluar_modelo(pipeline, nombre_modelo):
    print(f"\n{'='*40}")
    print(f"EVALUACIÓN: {nombre_modelo}")
    print(f"{'='*40}")
    
    y_pred = pipeline.predict(X_val)
    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    
    print(classification_report(y_val, y_pred))
    
    roc_auc = roc_auc_score(y_val, y_pred_proba)
    print(f"ROC-AUC Score: {roc_auc:.4f}\n")
    
    # Matriz de confusión
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Cancelado', 'Cancelado'])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Matriz de Confusión - {nombre_modelo}')
    plt.show()

# Evaluar ambos modelos
evaluar_modelo(pipeline_lr, "Regresión Logística")
evaluar_modelo(pipeline_rf, "Random Forest")


## ¿Por qué los Pipelines son geniales?
### Sobre `fit_transform` vs `transform` 

En Scikit-Learn, los preprocesadores (como `StandardScaler` o `OneHotEncoder`) tienen dos pasos:

1. __`fit` (Aprender):__ El algoritmo mira los datos y "aprende" sus parámetros. Por ejemplo, el `StandardScaler` calcula la media y la desviación estándar de la columna `adr`. El `OneHotEncoder` mira qué categorías únicas existen en la columna `meal`.
2. __`transform` (Aplicar):__ Usa lo que aprendió en el `fit` para modificar los datos.

__La Regla de Oro para evitar Data Leakage:__

- __Al set de Entrenamiento (Train):__ Le aplicamos `fit` (para que aprenda) y luego `transform` (para que se modifique). Scikit-Learn tiene una función combinada más rápida llamada __`fit_transform()`__ que hace ambas cosas a la vez.
- __Al set de Validación o Test:__ __NUNCA__ le aplicamos `fit`. Si lo hiciéramos, el escalador aprendería la media del futuro, ¡y eso es trampa! A estos sets __solo se les aplica `transform()`__, usando lo que el modelo ya aprendió del Train.

__¿Por qué usamos `Pipeline`?__ Porque el `Pipeline` hace todo esto automáticamente por ti. Cuando tú escribes `pipeline.fit(X_train, y_train)`, internamente el Pipeline hace:

1. `X_train_procesado = preprocessor.fit_transform(X_train)`
2. `modelo.fit(X_train_procesado, y_train)`

Y cuando escribes `pipeline.predict(X_val)`, internamente hace:

1. `X_val_procesado = preprocessor.transform(X_val)` *(¡Nota que aquí solo usa transform!)*
2. `modelo.predict(X_val_procesado)`

Por eso usar Pipelines es la forma más segura y profesional de programar modelos.


## ¿Cómo evalúas tu modelo?

Cuando tienes un modelo de clasificación entrenado (ya sea Regresión Logística o Random Forest), este puede darte dos tipos de respuestas ante un dato nuevo (`X_val`): la decisión final o el nivel de seguridad de esa decisión.

### 1. `y_pred = pipeline.predict(X_val)`
Esta línea le pide al modelo que tome una **decisión binaria final (blanco o negro)**.
*   El modelo mira cada reserva en `X_val` y dice: `"¿Se cancela (1) o no se cancela (0)?"`
*   Por defecto, el modelo usa un umbral de corte del 50% (0.5). Si cree que hay un 51% de probabilidad de que se cancele, devuelve un `1`. Si cree que hay un 49%, devuelve un `0`.
*   **¿Para qué sirve `y_pred`?** Se usa para calcular métricas de decisión dura, como el *Accuracy* (Exactitud), *Precision*, *Recall*, *F1-Score*, y para dibujar la **Matriz de Confusión** (cuántos falsos positivos y falsos negativos cometió).

### 2. `y_pred_proba = pipeline.predict_proba(X_val)[:, 1]`
Esta línea le pide al modelo que te diga **qué tan seguro está (probabilidad)**.
*   `predict_proba` devuelve una matriz con las probabilidades para cada clase. Para cada reserva, te da dos números: `[probabilidad_de_0, probabilidad_de_1]`. Por ejemplo: `[0.20, 0.80]` significa que está 80% seguro de que se cancelará.
*   El `[:, 1]` al final es una forma de decirle a Python: *"De todas las filas, dame solo la columna del índice 1"*. El índice 1 corresponde a la probabilidad de la clase positiva (es decir, la probabilidad de que **sí se cancele**, que es lo que nos interesa predecir).
*   **¿Para qué sirve `y_pred_proba`?** Se usa exclusivamente para calcular la métrica **ROC-AUC**. 
    *   El ROC-AUC no evalúa si el modelo acertó usando el corte del 50%. El ROC-AUC evalúa **la capacidad del modelo para ordenar a los clientes**. Si el modelo le da una probabilidad del 90% a alguien que sí canceló, y una del 10% a alguien que no canceló, el ROC-AUC será altísimo, sin importar dónde pongas el umbral de corte.

### En resumen:
*   `predict()`: Te da `[0, 1, 1, 0]`. Lo usas para el *Classification Report* y la *Matriz de Confusión*.
*   `predict_proba()[:, 1]`: Te da `[0.12, 0.85, 0.60, 0.05]`. Lo usas para calcular el *ROC-AUC Score*.

# EVALUACIÓN: Random Forest
¡Este es un excelente reporte para analizar! Vamos a desglosarlo parte por parte para que entiendas exactamente qué te está diciendo el Random Forest sobre cómo predice las cancelaciones.

Primero, recordemos qué significa cada fila (clase):
*   **Clase 0**: Reservas que **NO se cancelaron** (el cliente fue al hotel).
*   **Clase 1**: Reservas que **SÍ se cancelaron** (nuestro objetivo a predecir).

---

### 1. Las métricas por clase (Las columnas)

**Support (Soporte):**
Es la cantidad real de casos que hay en este set de validación.
*   Había **9299** reservas que no se cancelaron.
*   Había **3811** reservas que sí se cancelaron.
*(Nota que los datos están desbalanceados, hay muchas más reservas exitosas que canceladas).*

**Recall (Sensibilidad): ¡La métrica más importante aquí!**
*   **Para la clase 1 (0.72):** Significa que de las 3811 cancelaciones reales que hubo, **el modelo logró detectar el 72%**. ¡Esto es muy bueno! Atrapó a casi 3 de cada 4 personas que iban a cancelar. Esto se logró gracias a que usamos `class_weight='balanced'`.
*   **Para la clase 0 (0.79):** De todas las personas que realmente fueron al hotel, el modelo reconoció correctamente al 79%.

**Precision (Precisión):**
*   **Para la clase 1 (0.59):** Significa que de todas las veces que el modelo levantó la mano y dijo *"¡Alerta, este cliente va a cancelar!"*, **acertó el 59% de las veces**. El otro 41% fueron "Falsos Positivos" (clientes que el modelo creyó que cancelarían, pero al final sí fueron al hotel).
*   *Nota de negocio:* En hoteles, a veces es preferible tener una precisión del 59% y un recall alto del 72%. Es mejor prepararse para una cancelación que no ocurre (falso positivo), que ser tomado por sorpresa por una cancelación que no viste venir (falso negativo).

**F1-Score:**
Es simplemente un promedio matemático (promedio armónico) entre la Precisión y el Recall. Como para la clase 1 la precisión es 0.59 y el recall es 0.72, su F1-Score es **0.65**. Resume el rendimiento en un solo número.

---

### 2. Las métricas globales

**Accuracy (Exactitud = 0.77):**
De las 13,110 reservas totales, el modelo **acertó el 77% de las veces**. (Dijo "cancela" y canceló, o dijo "no cancela" y no canceló).

*(Ignora "macro avg" y "weighted avg", son solo promedios de las métricas de arriba y no aportan mucho valor de negocio).*

---

### 3. ROC-AUC Score: 0.8457
¡Este es un número **muy bueno**! 
El ROC-AUC va de 0.50 (el modelo tira una moneda al aire) a 1.0 (el modelo es adivino perfecto). 
Un puntaje de **0.8457** significa que el modelo tiene una excelente capacidad para separar a los clientes. 

**¿Cómo se lee en la vida real?**
Si tomas al azar a un cliente que *sí canceló* y a un cliente que *no canceló*, hay un **84.5% de probabilidad** de que el modelo le haya asignado un porcentaje de riesgo más alto al cliente que realmente canceló.

### Conclusión de este modelo:
Es un modelo muy sólido. Gracias al balanceo de clases, no es "tímido": se arriesga a predecir cancelaciones y logra atrapar el 72% de ellas, aunque a costa de equivocarse un poco (falsos positivos) cuando predice que alguien va a cancelar. Para un problema de demanda hotelera, ¡es un comportamiento muy deseable!

# Random Forest vs Regresión Logística

¡Esta es la clásica y hermosa paradoja del Machine Learning en problemas de negocio! Tienes toda la razón en tu observación: **La Regresión Logística tiene un Recall brutal (0.87) pero un ROC-AUC más bajo (0.7907) que el Random Forest (0.8457).**

Vamos a desmenuzar por qué pasa esto y cuál es mejor.

### ¿Por qué la Regresión Logística tiene tanto Recall (0.87)?
Mira la **Precisión de la clase 1 (0.44)**. 
La Regresión Logística está actuando como una alarma de incendios hiper-sensible. Ante la más mínima duda, grita: *"¡Va a cancelar!"*.
*   **Lo bueno (Recall 0.87):** Al gritar tanto, logra atrapar al 87% de los que realmente iban a cancelar. ¡Casi no se le escapa nadie!
*   **Lo malo (Precision 0.44):** Grita tantas veces en falso, que de todas las veces que dice que alguien va a cancelar, **se equivoca más de la mitad de las veces (56% son falsos positivos)**. 
*   Además, fíjate en el **Recall de la clase 0 (0.54)**: Apenas reconoce a la mitad de los clientes buenos. A la otra mitad los acusa injustamente de que van a cancelar.

### ¿Por qué el Random Forest tiene mejor ROC-AUC (0.8457)?
El ROC-AUC no evalúa los "gritos" (el corte de decisión en 50%), sino **qué tan bien el modelo ordena a los clientes de menor a mayor riesgo**.
El Random Forest es mucho más inteligente asignando probabilidades. Sabe distinguir mejor quién es realmente riesgoso y quién no. La Regresión Logística, aunque atrapa a muchos, hace una "ensalada" con las probabilidades, mezclando a clientes buenos y malos, por eso su ROC-AUC es menor.

---

### Entonces, ¿Cuál es mejor?

**El Random Forest es, matemáticamente y estructuralmente, un modelo MUCHO mejor.** (Mejor F1-Score, mejor Accuracy, mejor ROC-AUC).

Sin embargo, la decisión final depende de **cuánto le cuesta al hotel equivocarse**.

**Escenario A: El hotel usa las predicciones para llamar al cliente y confirmar (Costo bajo de Falso Positivo)**
Si la acción del hotel es simplemente mandar un WhatsApp diciendo: *"Hola, ¿sigue en pie tu reserva?"*, entonces la **Regresión Logística podría ser útil**. No importa molestar al 56% de clientes buenos (Falsos Positivos) si eso te permite salvar el 87% de las cancelaciones reales.

**Escenario B: El hotel usa las predicciones para hacer "Overbooking" (Costo altísimo de Falso Positivo)**
Si el hotel dice: *"El modelo dice que este cliente va a cancelar, así que le voy a vender su habitación a otra persona"*. ¡Cuidado! Si usas la Regresión Logística, el 56% de las veces venderás la habitación de alguien que **sí iba a ir**. Tendrás a dos personas peleando por una habitación en la recepción. En este caso, el **Random Forest es infinitamente mejor** porque su precisión es más alta (0.59) y comete menos de estos errores catastróficos.

### Mi recomendación:
**Quédate con el Random Forest.** 
Es un modelo mucho más equilibrado y robusto (F1-score de 0.65 vs 0.58). Si en el futuro necesitas que el Random Forest tenga un Recall de 0.87 como la Regresión, puedes lograrlo fácilmente **bajando el umbral de decisión** (ej. diciendo que cualquiera con >30% de probabilidad se considere cancelación), pero manteniendo la inteligencia superior del Random Forest para ordenar a los clientes.

# F1-Score
El **F1-Score** es una de las métricas más útiles e incomprendidas en Machine Learning. Para explicarlo de forma sencilla, imagínalo como el **"juez imparcial"** entre la Precisión y el Recall.

### El problema que resuelve el F1-Score
En clasificación, siempre hay una "guerra" entre la Precisión y el Recall. 
* Si quieres un **Recall perfecto (1.0)**, es fácil: simplemente dile al modelo que prediga que *TODOS* van a cancelar. Atraparás el 100% de las cancelaciones, pero tu Precisión será basura porque te equivocarás con todos los que sí iban a ir.
* Si quieres una **Precisión perfecta (1.0)**, es fácil: dile al modelo que solo prediga "Cancela" cuando esté 99.99% seguro. Tu precisión será perfecta, pero tu Recall será bajísimo porque se te escaparán casi todos.

Como no podemos mirar solo una de las dos métricas porque nos engañarían, necesitamos un número que las combine.

### ¿Qué es exactamente el F1-Score?
Es el **promedio armónico** entre la Precisión y el Recall. 

La fórmula matemática es:
`F1 = 2 * (Precision * Recall) / (Precision + Recall)`

**¿Por qué usamos el "promedio armónico" y no un promedio normal?**
Porque el promedio armónico castiga severamente los valores bajos. 
Imagina que un modelo tiene:
* Precisión = 1.0 (Perfecta)
* Recall = 0.01 (Pésimo)
Si hiciéramos un promedio normal, daría `0.50` (parece un modelo decente). 
Pero con el promedio armónico (F1-Score), el resultado da `0.019`. El F1-Score te grita: *"¡Este modelo es inútil porque una de sus métricas es un desastre!"*

### Cómo leer el F1-Score en tus resultados:

**Random Forest (Clase 1):**
* Precisión: 0.59
* Recall: 0.72
* **F1-Score: 0.65** 
*(Es un balance bastante saludable. El modelo es decentemente preciso y tiene muy buen recall).*

**Regresión Logística (Clase 1):**
* Precisión: 0.44
* Recall: 0.87
* **F1-Score: 0.58**
*(El F1-Score bajó respecto al Random Forest. El F1-Score está "castigando" a la Regresión Logística porque, aunque su Recall es altísimo, su Precisión de 0.44 es demasiado pobre).*

### En resumen:
El F1-Score es la métrica que debes mirar cuando quieres saber **"qué tan bueno es el modelo en general para esta clase"**, sin dejarte engañar por un modelo que hace trampa maximizando solo la Precisión o solo el Recall. Mientras más cerca de 1.0, mejor es el equilibrio del modelo.

In [ ]:
# Muestra el porcentaje de cada clase (Modelo Base)
pd.Series(y_val).value_counts(normalize=True)


Comparemos al "Recepcionista Perezoso" (Modelo Base) con nuestro "Random Forest":

__1. El Recepcionista Perezoso:__

- __Exactitud:__ 71%
- __Cancelaciones detectadas (Recall Clase 1):__ __0%__ (Se le escaparon las 3,811 cancelaciones. El hotel perdió muchísimo dinero por habitaciones vacías que no pudo revender).

__2. Nuestro Random Forest:__

- __Exactitud:__ 77% (Subió 6 puntos, parece poco, pero mira el siguiente punto).
- __Cancelaciones detectadas (Recall Clase 1):__ __72%__ (Logró identificar a 2,743 de esas 3,811 personas que iban a cancelar).

### Conclusión:

Tu intuición es 100% correcta: en datasets desbalanceados, mirar solo el `Accuracy` (Exactitud) es una trampa mortal, porque el 71% te lo regala el simple hecho de que la mayoría de la gente no cancela.

Por eso en el paso anterior te insistía tanto en mirar el __Recall (0.72)__, el __F1-Score__ y el __ROC-AUC__. Esas son las métricas que nos demuestran que el modelo realmente está usando inteligencia para encontrar la "aguja en el pajar" (las cancelaciones), en lugar de simplemente apostar por la opción más segura.
